# 🔬 Error Analysis v5 - Diagnosing Overfitting

**Problem:** Model overfits immediately (Best Iteration = 0)

**Hypothesis:** Mismatch between Train and Val distributions, or toxic rows in Val causing high error.

**Goal:** Understand WHY validation loss increases as model learns.

## 1. Setup

In [ ]:
!pip install polars xgboost seaborn -q

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style("whitegrid")
print("✅ Libraries loaded")

In [ ]:
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data/processed"

FILES = {
    "model": f"{BASE_PATH}/xgboost_final_model_v5_pure.json",
    "train": f"{BASE_PATH}/train_pure_v5.parquet",
    "val_raw": f"{BASE_PATH}/test_ready_for_xgboost.parquet"
}

for name, path in FILES.items():
    exists = "✅" if os.path.exists(path) else "❌"
    print(f"{exists} {name}: {path}")

## 2. Load Resources

In [ ]:
# Load Model
print("📂 Loading model...")
model = xgb.Booster()
model.load_model(FILES["model"])
print("✅ Model loaded")

# Load Training Data (already cleaned)
print("\n📂 Loading training data (pure)...")
df_train = pl.read_parquet(FILES["train"])
print(f"✅ Train: {df_train.shape[0]:,} rows")

# Load Validation Data (raw)
print("\n📂 Loading validation data (raw)...")
df_val_raw = pl.read_parquet(FILES["val_raw"])
print(f"✅ Val Raw: {df_val_raw.shape[0]:,} rows")

In [ ]:
# Define toxic mask
def get_toxic_mask():
    return (
        ((pl.col("CGM") < 110) & (pl.col("cgm_delta_15min") < -5) & (pl.col("carbs") == 0)) |
        ((pl.col("CGM") < 70) & (pl.col("carbs") == 0)) |
        ((pl.col("cgm_delta_15min") < -15) & (pl.col("carbs") == 0)) |
        ((pl.col("CGM") >= 70) & (pl.col("CGM") < 100) & (pl.col("cgm_delta_15min").abs() < 3) & (pl.col("carbs") == 0))
    )

# Apply same cleaning to validation
df_val = df_val_raw.filter(~get_toxic_mask())
n_dropped = len(df_val_raw) - len(df_val)
print(f"\n🧹 Cleaned Val: {len(df_val):,} rows (dropped {n_dropped:,})")

## 3. Generate Predictions

In [ ]:
FEATURES = [
    'CGM', 'carbs', 'time_sin', 'time_cos', 'is_weekend',
    'cgm_delta_5min', 'cgm_delta_15min', 'basal',
    'age', 'weight', 'gender_encoded', 'insulin_type_encoded'
]

available = [f for f in FEATURES if f in df_val.columns]
print(f"Features: {len(available)}")

# Prepare validation data
X_val = df_val.select(available).to_numpy().astype(np.float32)
dval = xgb.DMatrix(X_val, feature_names=available)

# Predict
print("🔮 Generating predictions...")
preds = model.predict(dval)
print(f"✅ Predictions: {len(preds):,}")

In [ ]:
# Create residuals DataFrame
df_res = df_val.with_columns([
    pl.Series(name="Pred_Bolus", values=preds),
]).with_columns([
    (pl.col("bolus") - pl.col("Pred_Bolus")).abs().alias("Abs_Error"),
    (pl.col("bolus") - pl.col("Pred_Bolus")).alias("Error")
])

print(f"✅ Residuals DataFrame: {df_res.shape}")

## 4. Statistical Forensics

### 4.1 Class Imbalance Check

In [ ]:
print("📊 CLASS IMBALANCE CHECK")
print("=" * 50)

# Training
train_zero = df_train.filter(pl.col("bolus") == 0).shape[0]
train_pos = df_train.filter(pl.col("bolus") > 0).shape[0]
train_ratio = train_zero / train_pos if train_pos > 0 else 0

# Validation
val_zero = df_val.filter(pl.col("bolus") == 0).shape[0]
val_pos = df_val.filter(pl.col("bolus") > 0).shape[0]
val_ratio = val_zero / val_pos if val_pos > 0 else 0

print(f"\nTRAINING:")
print(f"   Zero bolus:     {train_zero:,} ({train_zero/len(df_train)*100:.1f}%)")
print(f"   Positive bolus: {train_pos:,} ({train_pos/len(df_train)*100:.1f}%)")
print(f"   Ratio (0/+):    {train_ratio:.2f}")

print(f"\nVALIDATION:")
print(f"   Zero bolus:     {val_zero:,} ({val_zero/len(df_val)*100:.1f}%)")
print(f"   Positive bolus: {val_pos:,} ({val_pos/len(df_val)*100:.1f}%)")
print(f"   Ratio (0/+):    {val_ratio:.2f}")

print(f"\n⚠️ RATIO DIFFERENCE: {abs(train_ratio - val_ratio):.2f}")
if abs(train_ratio - val_ratio) > 1:
    print("   WARNING: Significant distribution mismatch!")

### 4.2 Distribution Statistics

In [ ]:
print("📊 BOLUS DISTRIBUTION STATS")
print("=" * 50)

print(f"\n{'Metric':<20} {'Train':>15} {'Val':>15}")
print("-" * 50)
print(f"{'Mean':<20} {df_train['bolus'].mean():>15.4f} {df_val['bolus'].mean():>15.4f}")
print(f"{'Std':<20} {df_train['bolus'].std():>15.4f} {df_val['bolus'].std():>15.4f}")
print(f"{'Median':<20} {df_train['bolus'].median():>15.4f} {df_val['bolus'].median():>15.4f}")
print(f"{'Max':<20} {df_train['bolus'].max():>15.4f} {df_val['bolus'].max():>15.4f}")
print(f"{'Q75':<20} {df_train['bolus'].quantile(0.75):>15.4f} {df_val['bolus'].quantile(0.75):>15.4f}")
print(f"{'Q95':<20} {df_train['bolus'].quantile(0.95):>15.4f} {df_val['bolus'].quantile(0.95):>15.4f}")

if df_val['bolus'].mean() > df_train['bolus'].mean() * 1.5:
    print("\n⚠️ Val mean is 50%+ higher than Train - Val is HARDER!")

### 4.3 Top 20 Worst Errors

In [ ]:
print("🚨 TOP 20 WORST ERRORS (Validation)")
print("=" * 70)

cols_to_show = ['CGM', 'carbs', 'cgm_delta_15min', 'bolus', 'Pred_Bolus', 'Abs_Error']
available_cols = [c for c in cols_to_show if c in df_res.columns]

worst_20 = df_res.sort("Abs_Error", descending=True).head(20)
print(worst_20.select(available_cols))

# Analysis
print("\n📋 WORST ERROR ANALYSIS:")
print(f"   Max Error: {worst_20['Abs_Error'].max():.2f} U")
print(f"   Mean True Bolus in worst 20: {worst_20['bolus'].mean():.2f} U")
print(f"   Mean Pred Bolus in worst 20: {worst_20['Pred_Bolus'].mean():.2f} U")

In [ ]:
# Check if worst errors are from low CGM scenarios
print("\n🔍 WORST ERROR BREAKDOWN:")

worst_low_cgm = worst_20.filter(pl.col("CGM") < 100).shape[0]
worst_high_cgm = worst_20.filter(pl.col("CGM") > 200).shape[0]
worst_no_carbs = worst_20.filter(pl.col("carbs") == 0).shape[0]

print(f"   Low CGM (<100): {worst_low_cgm}/20")
print(f"   High CGM (>200): {worst_high_cgm}/20")
print(f"   No Carbs: {worst_no_carbs}/20")

### 4.4 False Positive / False Negative Analysis

In [ ]:
print("⚠️ FALSE POSITIVE / FALSE NEGATIVE ANALYSIS")
print("=" * 50)

# False Positive: Truth=0, Pred > 0.5
fp = df_res.filter((pl.col("bolus") == 0) & (pl.col("Pred_Bolus") > 0.5))
n_fp = len(fp)
n_truth_zero = df_res.filter(pl.col("bolus") == 0).shape[0]
fp_rate = n_fp / n_truth_zero * 100 if n_truth_zero > 0 else 0

print(f"\n🔴 FALSE POSITIVE (Truth=0, Pred>0.5):")
print(f"   Count: {n_fp:,} / {n_truth_zero:,}")
print(f"   Rate: {fp_rate:.2f}%")

# False Negative: Truth>2, Pred < 0.5
fn = df_res.filter((pl.col("bolus") > 2) & (pl.col("Pred_Bolus") < 0.5))
n_fn = len(fn)
n_truth_high = df_res.filter(pl.col("bolus") > 2).shape[0]
fn_rate = n_fn / n_truth_high * 100 if n_truth_high > 0 else 0

print(f"\n🔵 FALSE NEGATIVE (Truth>2U, Pred<0.5):")
print(f"   Count: {n_fn:,} / {n_truth_high:,}")
print(f"   Rate: {fn_rate:.2f}%")

print(f"\n📊 ERROR METRICS:")
print(f"   MAE: {df_res['Abs_Error'].mean():.4f}")
print(f"   RMSE: {np.sqrt((df_res['Error']**2).mean()):.4f}")

### 4.5 Show False Negative Examples

In [ ]:
if n_fn > 0:
    print("🔵 FALSE NEGATIVE EXAMPLES (Missed High Doses):")
    print("=" * 60)
    print(fn.select(available_cols).head(10))
else:
    print("✅ No False Negatives (Truth>2U, Pred<0.5)")

## 5. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Train vs Val Bolus Distribution
ax1 = axes[0, 0]
ax1.hist(df_train['bolus'].to_numpy(), bins=50, alpha=0.5, label='Train', color='blue')
ax1.hist(df_val['bolus'].to_numpy(), bins=50, alpha=0.5, label='Val', color='red')
ax1.set_xlabel('Bolus (U)')
ax1.set_ylabel('Count')
ax1.set_title('Train vs Val Bolus Distribution')
ax1.legend()
ax1.set_xlim(0, 15)

# 2. Prediction vs Truth Scatter
ax2 = axes[0, 1]
sample_size = min(5000, len(df_res))
sample = df_res.sample(sample_size, seed=42)
ax2.scatter(sample['bolus'].to_numpy(), sample['Pred_Bolus'].to_numpy(), alpha=0.3, s=5)
ax2.plot([0, 15], [0, 15], 'r--', label='Perfect')
ax2.set_xlabel('True Bolus')
ax2.set_ylabel('Predicted Bolus')
ax2.set_title('Prediction vs Truth (Sample)')
ax2.legend()

# 3. Error Distribution
ax3 = axes[1, 0]
ax3.hist(df_res['Abs_Error'].to_numpy(), bins=50, color='orange', edgecolor='black')
ax3.axvline(df_res['Abs_Error'].mean(), color='red', linestyle='--', label=f'Mean: {df_res["Abs_Error"].mean():.2f}')
ax3.set_xlabel('Absolute Error')
ax3.set_ylabel('Count')
ax3.set_title('Error Distribution')
ax3.legend()

# 4. Error by CGM Range
ax4 = axes[1, 1]
cgm_bins = [(0, 70), (70, 100), (100, 140), (140, 180), (180, 250), (250, 600)]
bin_labels = ['<70', '70-100', '100-140', '140-180', '180-250', '>250']
bin_errors = []

for low, high in cgm_bins:
    subset = df_res.filter((pl.col('CGM') >= low) & (pl.col('CGM') < high))
    if len(subset) > 0:
        bin_errors.append(subset['Abs_Error'].mean())
    else:
        bin_errors.append(0)

ax4.bar(bin_labels, bin_errors, color='teal')
ax4.set_xlabel('CGM Range (mg/dL)')
ax4.set_ylabel('Mean Absolute Error')
ax4.set_title('Error by CGM Range')

plt.tight_layout()
plt.show()

## 6. Summary & Recommendations

In [ ]:
print("\n" + "=" * 60)
print("📋 ERROR ANALYSIS SUMMARY")
print("=" * 60)

print(f"""
DISTRIBUTION ANALYSIS:
   Train Mean Bolus: {df_train['bolus'].mean():.4f}
   Val Mean Bolus:   {df_val['bolus'].mean():.4f}
   Train Zero Ratio: {train_ratio:.2f}
   Val Zero Ratio:   {val_ratio:.2f}

ERROR METRICS:
   MAE: {df_res['Abs_Error'].mean():.4f}
   Max Error: {df_res['Abs_Error'].max():.2f}

FALSE POSITIVE/NEGATIVE:
   False Positive Rate: {fp_rate:.2f}%
   False Negative Rate: {fn_rate:.2f}%
""")

print("\n💡 RECOMMENDATIONS:")
if val_ratio < train_ratio * 0.8:
    print("   ⚠️ Val has MORE positive cases - Consider rebalancing")
if df_res['Abs_Error'].mean() > 1.0:
    print("   ⚠️ High MAE - Consider deeper feature engineering")
if fn_rate > 10:
    print("   ⚠️ High Miss Rate - Lower threshold or add safety rules")
if fp_rate > 20:
    print("   ⚠️ High False Positive - Increase threshold")

print("\n" + "=" * 60)